<a href="https://colab.research.google.com/github/goelnikhils-lgtm/languagemodels/blob/main/Embeddings_Reversability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Demonstrating embedding transformation to prevent reversibility while preserving similarity

import numpy as np
from sklearn.random_projection import GaussianRandomProjection
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import os

# Step 1: Generate synthetic embeddings (1000 samples, 128 dimensions)
np.random.seed(42)
original_embeddings = np.random.randn(1000, 128)

# Step 2: Apply random projection to reduce dimensionality (e.g., to 64 dimensions)
projector = GaussianRandomProjection(n_components=64, random_state=42)
projected_embeddings = projector.fit_transform(original_embeddings)

# Step 3: Inject small Gaussian noise into projected embeddings
noise = np.random.normal(loc=0.0, scale=0.05, size=projected_embeddings.shape)
noisy_embeddings = projected_embeddings + noise

# Step 4: Compare cosine similarity before and after transformation
# Select 10 random pairs of indices
indices = np.random.choice(range(1000), size=(10, 2), replace=False)

original_similarities = []
transformed_similarities = []

for i, j in indices:
    orig_sim = cosine_similarity([original_embeddings[i]], [original_embeddings[j]])[0][0]
    trans_sim = cosine_similarity([noisy_embeddings[i]], [noisy_embeddings[j]])[0][0]
    original_similarities.append(orig_sim)
    transformed_similarities.append(trans_sim)

# Plot similarity comparison
plt.style.use('seaborn-v0_8')
plt.figure(figsize=(8, 6))
plt.plot(original_similarities, label='Original Cosine Similarity', marker='o')
plt.plot(transformed_similarities, label='Transformed Cosine Similarity', marker='x')
plt.title('Cosine Similarity Before and After Embedding Transformation')
plt.xlabel('Sample Pair Index')
plt.ylabel('Cosine Similarity')
plt.legend()
plt.grid(True)
os.makedirs("/mnt/data", exist_ok=True)
plt.savefig("/mnt/data/similarity_comparison.png")
plt.close()

# Step 5: Attempt reconstruction using pseudo-inverse
pseudo_inverse = np.linalg.pinv(projector.components_.T)
reconstructed_embeddings = noisy_embeddings @ pseudo_inverse

# Compute reconstruction error (mean squared error)
reconstruction_error = np.mean((original_embeddings - reconstructed_embeddings) ** 2)

print(f"Applied random projection and noise to synthetic embeddings.")
print(f"Mean squared reconstruction error (original vs reconstructed): {reconstruction_error:.4f}")
print("Cosine similarity before and after transformation plotted and saved as 'similarity_comparison.png'.")